# BERT-Base uncased — DIMER E2E continued-pre-training tutorial: masked-token prediction on paper abstracts (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bert-masked-lm-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bert-masked-lm-pipeline/blob/main/tutorials/bert_masked_lm_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--bert%2Fbert--base--uncased-ffcc4d?style=flat)](https://huggingface.co/google-bert/bert-base-uncased) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fbert-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/bert) [![arXiv](https://img.shields.io/badge/arXiv-1810.04805-b31b1b.svg)](https://arxiv.org/abs/1810.04805)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** masked-language modelling (fill-mask: ranked candidates for one `[MASK]`), sentence embeddings (768-d, CLS or mean pooled, L2-normalised) and bounded continued masked-language-model training of the last encoder layers on a text corpus, measured by held-out masked-token perplexity and top-k accuracy, using the pinned BERT-Base uncased weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/bert_masked_lm_pipeline/`, at revision `3c7ff9f8a41f`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `86b5e0934494bd15c9632b12f734a8a67f723594` (~441 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned BERT-Base uncased snapshot (safetensors, 440 MB), fetches the three digest-pinned SciTLDR-A files from the project repository (5.5 MB, no credential), reads the paper abstracts and draws 300 / 50 / 100 training, validation and test documents from the release's own paper-disjoint members, fills one mask in an unseen abstract's opening sentence and embeds two test abstracts through the inference contracts with an input manifest and a rejection probe, scores the frozen model on the test abstracts by masked-token perplexity and top-1 / top-5 accuracy at seeded masks beside the add-one unigram floor, runs a bounded continued masked-language-model training of the last four encoder layers with validation-perplexity epoch selection, scores the held-out split again, fills the same masks and re-embeds the same abstracts with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about four minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own text corpus as a CSV (columns `id`, `text`), a JSON array or JSONL file of `{{id, text}}` records, or a plain `.txt` file in which every blank-line-separated paragraph is one document. It passes through the same validation, seeded text-disjoint split, unigram floor, frozen scoring, fine-tuning, held-out evaluation, cloze and embedding checks, artifact export and reload-parity cells as the SciTLDR sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the WordPiece tokenizer lower-cases the text and one forward pass of the 12-layer bidirectional encoder runs; `fill_mask` reads the output-vocabulary logits at the single `[MASK]` position through the masked-LM head and ranks them by a softmax over the 30,522-token vocabulary, while `embed` takes the encoder's last hidden states and pools them to one 768-d vector per text (the `[CLS]` position, or the attention-masked mean) and L2-normalises it. The carried pipeline module adds manifest verification, input validation and ceilings (over-long texts are rejected, not truncated), the two task methods and fixed output contracts. **The fill-mask `score` is not a calibrated probability** (it is a softmax ranking signal), and **embeddings are representations, not predictions**; the pipeline ships no threshold for either.

What this notebook adds to inference is **continued pre-training measured by the model's own objective**. The dataset is real: SciTLDR-A (Cachola et al., 2020; Apache-2.0) ships the abstracts of 3,229 computer-science papers as three digest-pinned JSON-Lines files fetched from the project repository at a pinned commit; only the abstract text is used, so every record is one document and no label is needed. The carried `metrics.py` masks a **seeded 15 %** of each held-out abstract's WordPiece tokens — the same positions for every model scored under the same seed — and reads the negative log-likelihood and the rank of the original token at every masked position, aggregated into **masked perplexity**, **bits per masked token** and **top-1 / top-5 accuracy**; an **add-one unigram model** fitted on the training tokens is the floor a model that ignores context reaches. The fine-tuning question is whether a bounded masked-language-model adaptation of the last encoder layers on 300 abstracts lowers that perplexity on abstracts the model has not seen. Nothing here is a quality claim: a lower masked perplexity says the adapted encoder predicts missing words of paper abstracts better, not that its embeddings are better for your task.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real corpus and validate and split it without leakage; run fill-mask and embedding through the public API and read the candidate and vector contracts correctly; read masked perplexity and top-k accuracy beside a unigram floor and understand what they do and do not measure; run a bounded continued masked-language-model training with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; compare cloze candidates and embedding similarities before and after; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** classification or other supervised heads, next-sentence prediction, multi-mask filling, raw-logit access, text generation (BERT is an encoder), cased or non-English text (the checkpoint is uncased English), contrastively trained sentence similarity (the `qwen3-embedding-pipeline` sibling covers retrieval-grade embeddings), full-model or embedding-table training, any similarity or downstream-task score, and any claim that a SciTLDR abstract split stands in for your corpus. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate: the build record measured about 4 s to load and digest-verify the 440 MB snapshot, about 7 s to score the 100-abstract test split (2,983 masked positions) and about 102 s per training epoch over 300 abstracts plus a 50-abstract validation pass per epoch. The pinned `torch==2.14.0` install and the 440 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what masked-language modelling is; what a softmax over a vocabulary is and why it is not a calibrated probability; what perplexity is (the exponential of the mean negative log-likelihood) and why a masked perplexity is an intrinsic number, not a judgement of embedding quality; what cosine similarity between unit vectors means.
- **Data contract:** records are `{{id, text}}` — one document of the domain, 1..4,000 characters and at most 512 WordPiece tokens including `[CLS]`/`[SEP]` (a longer record is refused, not truncated, everywhere), ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; texts are de-duplicated case-insensitively before splitting so the same document never sits in two splits; text is lower-cased by the tokenizer, so case carries no information. BYOD accepts CSV, JSON, JSONL or TXT in that shape.
- **Validation is structural, not semantic:** nothing checks that a record belongs to the domain you mean — an off-topic corpus is trained on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document collection is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches three pinned objects (`train.jsonl` 3,155,015 bytes, `dev.jsonl` 1,124,865 bytes, `test.jsonl` 1,204,107 bytes; SHA-256 `b222771d…` / `3191fa98…` / `fb42dd6c…`) from `raw.githubusercontent.com` at the pinned `allenai/scitldr` commit over HTTPS, each refused on any mismatch before it is read; SciTLDR is Apache-2.0 (Cachola et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `google-bert/bert-base-uncased` snapshot (~441 MB in total) at revision `86b5e0934494…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bert-masked-lm-pipeline',
    'repository_revision': '3c7ff9f8a41fe10f455c62da28ea94042425dcba',
    'embedded_module': 'src/bert_masked_lm_pipeline/pipeline.py',
    'embedded_modules': ['src/bert_masked_lm_pipeline/metrics.py', 'src/bert_masked_lm_pipeline/pipeline.py', 'src/bert_masked_lm_pipeline/samples.py'],
    'module_sha256': 'c294da2096eb7043ee6982ec49447a1be1b62373398b39f89d4ad4eacd4acc0c',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bert_masked_lm_pipeline/` @ `3c7ff9f8a41f`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/bert_masked_lm_pipeline/metrics.py`

In [ ]:
"""Masked-language-model metrics over a held-out text corpus, seeded masking, and the unigram baseline.

A record is scored by masking a seeded 15 % of its WordPiece tokens (never `[CLS]`/`[SEP]`, at least one
token) and reading, at every masked position, the model's negative log-likelihood of the original token and
the original token's rank in the model's prediction. Those aggregate into **masked perplexity**
(`exp(mean NLL)` over all masked positions, token-weighted), **bits per masked token** (`mean NLL / ln 2`)
and **top-1 / top-5 accuracy** (the original token is the first / among the first five candidates). The
masks are a fixed function of the seed and the record's id, so the frozen and adapted models are scored on
exactly the same positions. The **unigram baseline** predicts every masked position with an add-one-smoothed
unigram distribution fitted on the training tokens — the floor a model that ignores context reaches.
"""

from __future__ import annotations

import hashlib
import math
import random
from collections import Counter
from collections.abc import Sequence
from typing import Any

DEFAULT_MASK_RATE = 0.15
METRIC_DEFINITIONS = {
    "perplexity": (
        "exp of the mean negative log-likelihood (natural log) of the original token at every masked "
        "position of the held-out records, all masked positions weighted equally; lower is better"
    ),
    "bits_per_token": "the same mean negative log-likelihood divided by ln 2",
    "mean_nll": "the mean negative log-likelihood at the masked positions in nats",
    "top1_accuracy": "fraction of masked positions whose original token is the model's first candidate",
    "top5_accuracy": "fraction of masked positions whose original token is among the first five candidates",
    "masking": (
        "a seeded 15 % of each record's WordPiece tokens (never [CLS]/[SEP], at least one token) replaced "
        "by [MASK] — the same positions for every model scored under the same seed"
    ),
}


def record_seed(record_id: str, seed: int) -> int:
    """A stable per-record seed so the masked positions depend only on the record and the base seed."""
    digest = hashlib.sha256(f"{seed}:{record_id}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


def mask_positions(n_tokens: int, *, rate: float = DEFAULT_MASK_RATE, seed: int) -> list[int]:
    """Sorted positions to mask among 1..n_tokens-2 (the interior; position 0 is [CLS], the last is [SEP])."""
    if not 0.0 < rate <= 1.0:
        raise ValueError("rate must be in (0, 1]")
    interior = n_tokens - 2
    if interior < 1:
        raise ValueError("a record needs at least one token between [CLS] and [SEP]")
    count = max(1, round(interior * rate))
    return sorted(random.Random(seed).sample(range(1, n_tokens - 1), count))


def masked_metrics(scores: Sequence[Sequence[tuple[float, int]]]) -> dict[str, Any]:
    """Aggregate per-record lists of (NLL in nats, rank of the original token, 1 = first) pairs."""
    if not scores:
        raise ValueError("no records to score")
    total = 0.0
    n_masked = 0
    top1 = 0
    top5 = 0
    for record_scores in scores:
        if not record_scores:
            raise ValueError("a record scored no masked positions")
        for nll, rank in record_scores:
            if rank < 1:
                raise ValueError("rank must be 1 or more")
            total += float(nll)
            n_masked += 1
            top1 += rank == 1
            top5 += rank <= 5
    mean_nll = total / n_masked
    return {
        "n_records": len(scores),
        "n_masked": n_masked,
        "mean_nll": mean_nll,
        "perplexity": math.exp(mean_nll),
        "bits_per_token": mean_nll / math.log(2),
        "top1_accuracy": top1 / n_masked,
        "top5_accuracy": top5 / n_masked,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def unigram_masked_scores(
    train_ids: Sequence[Sequence[int]], targets: Sequence[Sequence[int]], vocab_size: int
) -> list[list[tuple[float, int]]]:
    """(NLL, rank) of each masked original token under an add-one unigram model fitted on `train_ids`.
    The rank counts vocabulary entries with a strictly higher count plus one, so ties favour the target."""
    if vocab_size < 1:
        raise ValueError("vocab_size must be positive")
    counts: Counter[int] = Counter()
    for ids in train_ids:
        counts.update(int(t) for t in ids)
    total = sum(counts.values()) + vocab_size
    log_total = math.log(total)
    ordered = sorted(counts.values(), reverse=True)
    out = []
    for record_targets in targets:
        record = []
        for target in record_targets:
            count = counts.get(int(target), 0)
            higher = sum(1 for c in ordered if c > count)
            record.append((log_total - math.log(count + 1), higher + 1))
        out.append(record)
    return out


def unigram_baseline(
    train_ids: Sequence[Sequence[int]], targets: Sequence[Sequence[int]], vocab_size: int
) -> dict[str, Any]:
    """Masked metrics of the context-free unigram model — the floor a masked language model must beat."""
    result = masked_metrics(unigram_masked_scores(train_ids, targets, vocab_size))
    result["baseline"] = "add-one-smoothed unigram model fitted on the training tokens (ignores context)"
    return result

**Module 2/3:** `src/bert_masked_lm_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Masked-language modelling and sentence embeddings over the pinned ``google-bert/bert-base-uncased``.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. Two task methods: ``fill_mask`` (one ``[MASK]`` token ->
ranked vocabulary candidates) and ``embed`` (CLS or mean pooled, L2-normalised 768-d representations).

The adaptation contract (``evaluate``, ``unigram_baseline``, ``adapt``, ``save_artifact``, ``from_artifact``)
scores a validated ``{id, text}`` corpus by masked-token prediction at seeded positions, continues the
masked-language-model objective on it for the last encoder layers with validation-perplexity epoch selection,
and exports the trained tensors as a safetensors adapter bound to the pinned base weights. The two inference
methods are unchanged by it, but both read the adapted encoder once ``adapt`` or ``load_artifact`` has run.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "google-bert/bert-base-uncased"
MODEL_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "bert-base-uncased"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. 512 is max_position_embeddings in the pinned config.json and model_max_length in
# tokenizer_config.json; longer inputs are rejected (not truncated) so a caller never silently loses [MASK].
MAX_TEXT_TOKENS = 512
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard; ~4 chars per WordPiece token on English text
MAX_BATCH = 64  # texts per embed() call
MAX_TOP_K = 100
VOCAB_SIZE = 30522  # config.json vocab_size
HIDDEN_SIZE = 768  # config.json hidden_size
MASK_TOKEN = "[MASK]"
POOLINGS = ("cls", "mean")
DEFAULT_TOP_K = 5
PAD_TOKEN_ID = 0  # vocab.txt [PAD]
CLS_TOKEN_ID = 101  # vocab.txt [CLS]
SEP_TOKEN_ID = 102  # vocab.txt [SEP]
MASK_TOKEN_ID = 103  # vocab.txt [MASK]
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 109_514_298
ENCODER_LAYERS = 12  # config.json num_hidden_layers
DEFAULT_TRAINABLE_LAYERS = 4  # the last four encoder layers (28,351,488 parameters)
MAX_EVAL_RECORDS = 2_000
MAX_RECORDS_FIT = 20_000  # the unigram baseline may be fitted on a whole training split
MIN_SCORED_RECORDS = 50  # below this a scored corpus is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.bert-base-uncased.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_text(text: Any, name: str) -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_mask_count(text: str) -> str:
    """`fill_mask` accepts exactly one [MASK]; raise naming the count found."""
    if text.count(MASK_TOKEN) != 1:
        raise ValueError(f"text must contain exactly one {MASK_TOKEN}, found {text.count(MASK_TOKEN)}")
    return text


def _check_top_k(top_k: Any) -> int:
    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an int")
    if not 1 <= top_k <= MAX_TOP_K:
        raise ValueError(f"top_k must be between 1 and MAX_TOP_K={MAX_TOP_K}")
    return top_k


def _check_batch(texts: Any, pooling: Any) -> list[str]:
    """`embed`'s batch contract; raise naming the first violated ceiling."""
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a list of str, not a single string")
    if not 1 <= len(texts) <= MAX_BATCH:
        raise ValueError(f"texts must hold 1..MAX_BATCH={MAX_BATCH} items, got {len(texts)}")
    clean = [_check_text(t, f"texts[{i}]") for i, t in enumerate(texts)]
    if pooling not in POOLINGS:
        raise ValueError(f"pooling must be one of {POOLINGS}")
    return clean


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "sequence of non-empty str; every entry carrying a [MASK] token is also a fill_mask input, "
        "every entry is an embed input (one vector per text)"
    ),
    "batch": [1, MAX_BATCH],
    "text_chars": [1, MAX_TEXT_CHARS],
    "text_tokens": [1, MAX_TEXT_TOKENS],
    "top_k": [1, MAX_TOP_K],
    "pooling": list(POOLINGS),
    "mask_token": MASK_TOKEN,
    "masks_per_fill_mask_text": 1,
    "vocab_size": VOCAB_SIZE,
    "embedding_dim": HIDDEN_SIZE,
    "preprocessing": (
        "WordPiece tokenisation that lower-cases and strips accents; texts past MAX_TEXT_TOKENS are "
        "rejected, never truncated, so a [MASK] can never be silently lost; embed pools the last "
        "layer (cls position or attention-masked mean) and L2-normalises"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    *,
    top_k: int = DEFAULT_TOP_K,
    pooling: str = "cls",
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``texts`` is the batch ``embed`` would take; every entry that carries a ``[MASK]`` token is
    additionally checked against ``fill_mask``'s contract (exactly one mask, ``top_k`` in range) and
    marked in the manifest. Both capabilities' checks run through the same private functions the
    methods use — ``_check_batch``/``_check_text`` for ``embed``, ``_check_mask_count``/``_check_top_k``
    for ``fill_mask`` — so a rejection here is a rejection there. ``MAX_TEXT_TOKENS`` is enforced
    after tokenisation inside the pipeline and therefore cannot be observed at this stage.
    """
    checked = _check_batch(texts, pooling)
    _check_top_k(top_k)
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    inputs = []
    for i, text in enumerate(checked):
        masks = text.count(MASK_TOKEN)
        if masks:
            _check_mask_count(text)
        inputs.append(
            {
                "id": names[i] if names else f"text-{i}",
                "chars": len(text),
                "masks": masks,
                "fill_mask_input": bool(masks),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "top_k": top_k,
        "pooling": pooling,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], expected_tokens: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    Neither capability has a metric helper in this repository, so the verdict is always
    ``not-measurable`` (EVAL9). ``expected_tokens`` exists for interface parity with the fleet's
    other pipelines and is recorded in ``reason`` rather than scored: one author-expected token is
    an intent, not a labelled cloze set, and computing a hit rate from it would present a single
    observation as an accuracy. ``result`` is the ``fill_mask`` result; the embedding half is a
    representation and is covered by the same verdict.
    """
    candidates = result.get("candidates", [])
    supplied = expected_tokens is not None
    return {
        "task": "masked-language modelling (fill-mask) and sentence embedding",
        "score_semantics": (
            f"fill_mask `score` is a softmax over the {VOCAB_SIZE}-token vocabulary at the masked "
            "position — a ranking signal, not a calibrated probability, with argmax as the decision "
            f"rule and no shipped threshold; embed returns {HIDDEN_SIZE}-d unit vectors whose only "
            "meaning is cosine within the same model and pooling policy"
        ),
        "sample_kind": sample_kind,
        "n_candidates": len(candidates),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "the repository ships no metric helper for either capability"
            + (
                "; an expected token was supplied, but one author-expected token is an intent rather "
                "than a labelled cloze set, so scoring it would present a single observation as an accuracy"
                if supplied
                else "; the evaluated sample carries no gold tokens and no similarity labels"
            )
        ),
        "needs": (
            "for fill-mask, a labelled cloze set (sentence, mask position, gold token) over enough "
            "sentences to state a dispersion, scored with the caller's own top-1/top-k hit-rate code; "
            "for the embeddings, a judged similarity set (Spearman correlation) or a retrieval or "
            "clustering set with relevance labels (recall@k) — neither of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class BERTMaskedLMPipeline:
    """``_mask_runner`` maps one text to (vocab logits at the [MASK] position, n_tokens);
    ``_embed_runner`` maps texts to (last hidden states (N, T, 768), attention mask (N, T)); both injectable.
    ``_decode`` maps a token id to its string."""

    _mask_runner: Callable[[str], tuple[np.ndarray, int]]
    _embed_runner: Callable[[list[str]], tuple[np.ndarray, np.ndarray]]
    _decode: Callable[[int], str]
    device: str = "cpu"
    source: str = "injected"
    _encode: Callable[[str], list[int]] | None = field(default=None, repr=False)
    _mlm_scorer: Callable[[list[int], list[int], list[int]], list[tuple[float, int]]] | None = field(
        default=None, repr=False
    )
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BERTMaskedLMPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
            origin = "local-snapshot"
        elif allow_download:
            source, kwargs, origin = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, BertForMaskedLM

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BertForMaskedLM.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        mask_id = tokenizer.mask_token_id

        def mask_runner(text: str) -> tuple[np.ndarray, int]:
            batch = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            n_tokens = int(batch["input_ids"].shape[1])
            if n_tokens > MAX_TEXT_TOKENS:
                raise ValueError(f"text tokenises to {n_tokens} tokens; MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            position = (batch["input_ids"][0] == mask_id).nonzero().flatten()
            with torch.inference_mode():
                logits = model(**batch).logits[0, position[0]]
            return logits.float().cpu().numpy(), n_tokens

        def embed_runner(texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
            batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=False)
            if batch["input_ids"].shape[1] > MAX_TEXT_TOKENS:
                raise ValueError(f"a text tokenises past MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}")
            batch = batch.to(resolved_device)
            with torch.inference_mode():
                hidden = model.bert(**batch).last_hidden_state
            return hidden.float().cpu().numpy(), batch["attention_mask"].cpu().numpy()

        def encode(text: str) -> list[int]:
            return [int(i) for i in tokenizer(text, truncation=False)["input_ids"]]

        def mlm_scorer(ids: list[int], positions: list[int], targets: list[int]) -> list[tuple[float, int]]:
            """(NLL of the original token, its rank) at every masked position of one already-masked record."""
            input_ids = torch.tensor([ids], dtype=torch.long, device=resolved_device)
            with torch.inference_mode():
                logits = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids)).logits[0]
            rows = logits[positions].float()
            log_probs = torch.log_softmax(rows, dim=-1)
            target = torch.tensor(targets, dtype=torch.long, device=resolved_device)
            nll = -log_probs.gather(1, target[:, None])[:, 0]
            rank = (rows > rows.gather(1, target[:, None])).sum(dim=1) + 1
            return list(zip(nll.tolist(), rank.tolist(), strict=True))

        return cls(
            mask_runner,
            embed_runner,
            tokenizer.convert_ids_to_tokens,
            resolved_device,
            origin,
            _encode=encode,
            _mlm_scorer=mlm_scorer,
            _model=model,
            _tokenizer=tokenizer,
        )

    def fill_mask(self, text: str, top_k: int = DEFAULT_TOP_K) -> dict[str, Any]:
        """Rank candidates for exactly one ``[MASK]``; ``score`` is a softmax over the 30 522-token vocab."""
        text = _check_mask_count(_check_text(text, "text"))
        top_k = _check_top_k(top_k)
        logits, n_tokens = self._mask_runner(text)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (VOCAB_SIZE,):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({VOCAB_SIZE},)")
        shifted = np.exp(logits - logits.max())
        probs = shifted / shifted.sum()
        order = np.argsort(-probs, kind="stable")[:top_k]
        candidates = [
            {
                "token": self._decode(int(i)),
                "token_id": int(i),
                "score": float(probs[i]),
                "sequence": text.replace(MASK_TOKEN, self._decode(int(i)), 1),
            }
            for i in order
        ]
        return {
            "candidates": candidates,
            "top_k": top_k,
            "n_tokens": n_tokens,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def embed(self, texts: Sequence[str], pooling: str = "cls") -> dict[str, Any]:
        """L2-normalised 768-d representations: CLS token or attention-masked mean of the last layer."""
        clean = _check_batch(texts, pooling)
        hidden, mask = self._embed_runner(clean)
        hidden = np.asarray(hidden, dtype=np.float32)
        mask = np.asarray(mask, dtype=np.float32)
        if hidden.ndim != 3 or hidden.shape[0] != len(clean) or hidden.shape[2] != HIDDEN_SIZE:
            raise RuntimeError(f"backend returned {hidden.shape}, expected ({len(clean)}, T, {HIDDEN_SIZE})")
        if pooling == "cls":
            pooled = hidden[:, 0]
        else:
            counts = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
            pooled = (hidden * mask[:, :, None]).sum(axis=1) / counts
        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)
        return {
            "embeddings": normalized.tolist(),
            "dim": HIDDEN_SIZE,
            "pooling": pooling,
            "normalized": True,
            "n_tokens": [int(v) for v in mask.sum(axis=1)],
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def _record_ids(self, records: Sequence[Mapping[str, Any]]) -> list[list[int]]:
        """Tokenise validated records with [CLS]/[SEP]; a record is refused (never truncated) above
        MAX_TEXT_TOKENS or without at least one interior token."""
        if self._encode is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        out = []
        for record in records:
            ids = list(self._encode(record["text"]))
            if len(ids) > MAX_TEXT_TOKENS:
                raise ValueError(f"record {record['id']} has {len(ids)} tokens; ceiling is {MAX_TEXT_TOKENS}")
            if len(ids) < 3:
                raise ValueError(f"record {record['id']} has no token between [CLS] and [SEP]")
            out.append(ids)
        return out

    @staticmethod
    def _masked(
        records: Sequence[Mapping[str, Any]], ids: Sequence[Sequence[int]], *, rate: float, seed: int
    ) -> list[tuple[list[int], list[int], list[int]]]:
        """(masked ids, positions, original tokens) per record under the seeded per-record masking."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import mask_positions, record_seed` removed — names are kernel globals defined by the carried modules

        out = []
        for record, record_ids in zip(records, ids, strict=True):
            positions = mask_positions(len(record_ids), rate=rate, seed=record_seed(record["id"], seed))
            masked = list(record_ids)
            for position in positions:
                masked[position] = MASK_TOKEN_ID
            out.append((masked, positions, [record_ids[p] for p in positions]))
        return out

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, mask_rate: float = 0.15, seed: int = 0
    ) -> dict[str, Any]:
        """Masked-token prediction on a validated corpus: a seeded `mask_rate` of each record's interior
        tokens is replaced by [MASK] and the original tokens are scored by NLL and rank."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import masked_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if self._mlm_scorer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        scores = [
            self._mlm_scorer(masked, positions, targets)
            for masked, positions, targets in self._masked(
                checked, self._record_ids(checked), rate=mask_rate, seed=seed
            )
        ]
        metrics = masked_metrics(scores)
        metrics.update(
            {
                "mask_rate": mask_rate,
                "seed": seed,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def unigram_baseline(
        self,
        train: Sequence[Mapping[str, Any]],
        test: Sequence[Mapping[str, Any]],
        *,
        mask_rate: float = 0.15,
        seed: int = 0,
    ) -> dict[str, Any]:
        """The add-one unigram model fitted on `train` (interior tokens), scored at the same masked positions
        of `test` that `evaluate` uses under the same seed."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import unigram_baseline` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        train_checked = validate_dataset(train, min_records=1, max_records=MAX_RECORDS_FIT)["records"]
        test_checked = validate_dataset(test, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        train_ids = [ids[1:-1] for ids in self._record_ids(train_checked)]
        targets = [
            record_targets
            for _masked, _positions, record_targets in self._masked(
                test_checked, self._record_ids(test_checked), rate=mask_rate, seed=seed
            )
        ]
        result = unigram_baseline(train_ids, targets, VOCAB_SIZE)
        result.update({"mask_rate": mask_rate, "seed": seed})
        return result

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= ENCODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 1..{ENCODER_LAYERS}")
        model, _ = self._require_model()
        first = ENCODER_LAYERS - trainable_layers
        prefixes = tuple(f"bert.encoder.layer.{k}." for k in range(first, ENCODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        lr: float = 5e-5,
        batch_size: int = 8,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        mask_rate: float = 0.15,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded continued masked-language-model training on a validated text corpus.

        Only the last `trainable_layers` encoder layers train (4 by default; the word, position and
        token-type embeddings, the earlier layers, the pooler and the MLM head with its tied decoder stay
        frozen). Every epoch re-draws a seeded `mask_rate` of each record's interior tokens, replaces them
        with [MASK] and applies cross-entropy at those positions only; AdamW at a fixed learning rate,
        gradient clipping at 1.0, no scheduler; records over MAX_TEXT_TOKENS are refused, never truncated.
        Epoch 0 records the frozen model's validation metrics under the evaluation masking (`seed`); the
        epoch with the lowest validation masked perplexity is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        if not (0.0 < mask_rate <= 0.5):
            raise ValueError("mask_rate must be in (0, 0.5]")
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        train_ids = self._record_ids(train_checked)
        import torch

        torch.manual_seed(seed)
        model, _ = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            keep = ("perplexity", "bits_per_token", "top1_accuracy", "top5_accuracy", "n_masked")
            return {
                k: v
                for k, v in self.evaluate(val_checked, mask_rate=mask_rate, seed=seed).items()
                if k in keep
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_ppl = entry["val"]["perplexity"] if entry["val"] else math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            masked = self._masked(train_checked, train_ids, rate=mask_rate, seed=seed + 1_000_003 * epoch)
            order = torch.randperm(len(masked), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [masked[i] for i in order[start : start + batch_size]]
                width = max(len(ids) for ids, _p, _t in batch)
                input_ids = torch.full((len(batch), width), PAD_TOKEN_ID, dtype=torch.long)
                attention = torch.zeros((len(batch), width), dtype=torch.long)
                labels = torch.full((len(batch), width), -100, dtype=torch.long)
                for row, (ids, positions, targets) in enumerate(batch):
                    input_ids[row, : len(ids)] = torch.tensor(ids)
                    attention[row, : len(ids)] = 1
                    labels[row, positions] = torch.tensor(targets)
                out = model(
                    input_ids=input_ids.to(device),
                    attention_mask=attention.to(device),
                    labels=labels.to(device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["perplexity"] if entry["val"] else -math.inf
            if current < best_ppl or not entry["val"]:
                best_ppl = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "objective": "masked language modelling (continued pre-training)",
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "lowest validation masked perplexity"
            if val_checked
            else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "mask_rate": mask_rate,
            "n_train": len(train_checked),
            "n_train_tokens": sum(len(ids) - 2 for ids in train_ids),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted encoder-layer tensors as safetensors with a manifest naming the base."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("bert.encoder.layer."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable encoder-layer tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BERTMaskedLMPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/bert_masked_lm_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Text-corpus dataset contract for continued masked-language-model training: the pinned SciTLDR sample of
paper abstracts, validation, seeded splitting, BYOD loaders and CSV export.

The default corpus is **real** and away from BERT's BooksCorpus + Wikipedia pre-training distribution: the
abstracts of computer-science papers shipped by SciTLDR (Cachola et al., EMNLP Findings 2020; Apache-2.0). The
three `SciTLDR-A` JSON-Lines files are fetched one by one from the project repository at a pinned commit and
refused on any byte-size or SHA-256 mismatch; only the abstract text is used here. The frozen model's masked
perplexity on held-out abstracts is the number to beat, and the fine-tuning question is whether a bounded
continued-MLM adaptation of the last encoder layers lowers it on abstracts the model has not seen.

A record is ``{id, text}``: one document of the domain, no label and no reference — masked-token prediction
needs none.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTLDR-A (paper abstracts)"
CORPUS_RELEASE = "allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20"
CORPUS_BASE_URL = "https://raw.githubusercontent.com/allenai/scitldr/5ccad9c00a60ad75c9e04abf7f27d0f53f983b20/SciTLDR-Data/SciTLDR-A/"
CORPUS_FILES = {
    "train": ("train.jsonl", 3_155_015, "b222771d387be585cfdf5ae957b36757138415a352e0a3e3b23f73f87c3b1119"),
    "dev": ("dev.jsonl", 1_124_865, "3191fa98ccc09521332b7a1cd63b1930be4e8df125a235ccd31e40329709525e"),
    "test": ("test.jsonl", 1_204_107, "fb42dd6cd4f4a1928ae8a01a189456fbfe994a07e938bd49f68653933f6503c9"),
}
CORPUS_LICENSE = "Apache-2.0 (Cachola et al. 2020; allenai/scitldr)"
CORPUS_PAPERS = {"train": 1_992, "dev": 619, "test": 618}
DEFAULT_CACHE_DIR = Path("weights") / "scitldr"
MAX_SAMPLE_TEXT_CHARS = (
    2_000  # longer abstracts are left out of the sample (the ceiling is 512 WordPiece tokens)
)
MIN_SAMPLE_TEXT_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 300, "validation": 50, "test": 100}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the three pinned SciTLDR-A files (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the JSON-Lines members into abstract records keeping each SciTLDR `paper_id`."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        records = []
        for line in files[split].decode("utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            records.append(
                {
                    "id": f"{split}-{row['paper_id']}",
                    "text": " ".join(str(s).strip() for s in row["source"]),
                    "paper_id": str(row["paper_id"]),
                }
            )
        if len(records) != CORPUS_PAPERS[split]:
            raise ValueError(f"{split}: {len(records)} papers, expected {CORPUS_PAPERS[split]}")
        out[split] = records
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep records whose text is within the sample length window; drop repeated texts case-insensitively."""
    seen: set[str] = set()
    kept = []
    for record in records:
        text = str(record["text"])
        if not MIN_SAMPLE_TEXT_CHARS <= len(text) <= MAX_SAMPLE_TEXT_CHARS:
            continue
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)
        kept.append(dict(record))
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draws from the three SciTLDR members: training from `train`, validation from `dev`, test from
    `test` — the release's own paper-disjoint partition, re-checked on texts by `check_split_disjoint`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    source_of = {"train": "train", "validation": "dev", "test": "test"}
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        pool = filter_records(corpus[source_of[name]])
        if size > len(pool):
            raise ValueError(f"requested {size} {name} records but only {len(pool)} fit")
        rng.shuffle(pool)
        out[name] = [
            {"id": f"{name}-{i:04d}", "text": r["text"], "paper_id": r["paper_id"]}
            for i, r in enumerate(pool[:size])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/text")
    for key in ("id", "text"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, text = record["id"], record["text"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    if not isinstance(text, str):
        raise ValueError(f"{label}: text must be a string")
    if not text.strip():
        raise ValueError(f"{label}: text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{label}: text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    item = {"id": rid, "text": text.strip()}
    if "paper_id" in record:
        item["paper_id"] = str(record["paper_id"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text corpus; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    texts: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        texts.add(item["text"].lower())
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_texts": len(texts),
        "text_chars": {
            "min": min(len(r["text"]) for r in checked),
            "max": max(len(r["text"]) for r in checked),
        },
        "total_chars": sum(len(r["text"]) for r in checked),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["text"]] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased text appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["text"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(f"a text ({record['text'][:60]!r}…) appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD corpus into train/validation/test after de-duplicating texts."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["text"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, text}` records from a CSV (columns id, text), a JSON array or JSONL of such objects, or a
    plain `.txt` file in which every non-empty line (or blank-line-separated paragraph) is one record."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "text"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "text": r["text"]} for r in rows]
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    if suffix == ".txt":
        paragraphs = [p.strip() for p in text.replace("\r\n", "\n").split("\n\n") if p.strip()]
        return [{"id": f"doc-{i:05d}", "text": " ".join(p.split())} for i, p in enumerate(paragraphs)]
    raise ValueError("BYOD corpora must be .csv, .json, .jsonl or .txt")


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "text"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "text": record["text"]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `86b5e0934494…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bert-base-uncased",
  "modelId": "google-bert/bert-base-uncased",
  "revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 11356,
      "sha256": "43070e2d4e532684de521b885f385d0841030efa2b1a20bafb76133a5e1379c1"
    },
    {
      "path": "README.md",
      "bytes": 10517,
      "sha256": "9187b6018ea0010d884e78e098e328faa1b88b301570d0cce606bb35e4067e17"
    },
    {
      "path": "config.json",
      "bytes": 570,
      "sha256": "7160e1553ad2ca51d8c1cb066be533db31826e12d173824c1bb0cb1a4f187d20"
    },
    {
      "path": "coreml/fill-mask/float32_model.mlpackage/Manifest.json",
      "bytes": 617,
      "sha256": "4e9566ed44c3c401dada9ff243e0efbd570f9bc4e55dd7fdfb937cb23993c94c"
    },
    {
      "path": "model.safetensors",
      "bytes": 440449768,
      "sha256": "68d45e234eb4a928074dfd868cead0219ab85354cc53d20e772753c6bb9169d3"
    },
    {
      "path": "tokenizer.json",
      "bytes": 466062,
      "sha256": "ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 48,
      "sha256": "a025160ef0431f1a392f6f050c1310f4c5d9fb6f275932dbccba73c4d214bf10"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 441170446
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BERTMaskedLMPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the three pinned SciTLDR-A files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` flattens each JSON-Lines member into records whose `text` is the abstract's sentences joined by a space — titles and TLDRs are left unread. `build_sample_dataset` keeps abstracts of 200..2,000 characters, drops repeated texts, and draws 300 training documents from the `train` member, 50 validation documents from `dev` and 100 test documents from `test` by a seeded shuffle — the release's own paper-disjoint partition. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no text appears in two splits, and the training split is written to `outputs/bert_masked_lm_train.csv` in the shape BYOD expects.

Look for: 1,992 + 619 + 618 raw papers, three digests, splits 300 / 50 / 100, and four refusal probes — a duplicate id, an empty text, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_papers = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/scitldr'))
    raw_papers = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/bert_masked_lm_train.csv')
print({'data_source': data_source, 'raw_papers': raw_papers, 'splits': disjoint, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_texts': manifest['unique_texts'], 'text_chars': manifest['text_chars'], 'total_chars': manifest['total_chars'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {'id': train_records[0]['id'], 'text': train_records[0]['text'][:200] + '...'}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty text': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'missing field': [{'id': r['id']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Fill a mask and embed through the inference contracts

Before any adaptation, both inference contracts are exercised as they always were. A **cloze** is made from an unseen abstract's opening sentence by replacing its longest alphabetic word with `[MASK]` — the removed word is the *gold* word, which may be several WordPiece tokens, in which case no single candidate can match it (reported, never asserted). `validate_inputs` applies exactly the checks `fill_mask` and `embed` apply (text type and character ceiling, exactly one `[MASK]` for the cloze, `top_k` and `pooling` within their contracts) and returns an input manifest; the WordPiece ceiling `MAX_TEXT_TOKENS` (512) needs the real tokenizer and is enforced inside the pipeline, which **rejects, never truncates**. A two-mask input is validated too and its rejection recorded as a finding. `fill_mask` returns `top_k` candidates with a softmax `score` (a ranking signal, not a calibrated probability); `embed` returns one unit-norm 768-d vector per text under the requested pooling. Three unseen clozes and the embeddings of two test abstracts are kept as the *before* column for Section 9.

In [ ]:
import re
import time

TOP_K = 5  # @param {type:"integer"}
POOLING = 'mean'  # @param ["cls", "mean"]

def cloze_of(record):
    """The opening sentence with its longest alphabetic word replaced by [MASK]; returns (cloze, gold word)."""
    sentence = record['text'].split('. ')[0]
    words = [w for w in re.findall(r'[A-Za-z]+', sentence) if len(w) >= 4]
    gold = max(words, key=len)
    return re.sub(r'\b' + gold + r'\b', MASK_TOKEN, sentence, count=1), gold.lower()

if USE_BYOD:
    unseen_records = [{**r, 'id': f'unseen-{i:02d}'} for i, r in enumerate(test_records[2:5])]
else:
    used = {r['text'].lower() for part in splits.values() for r in part}
    unseen_records = [{**r, 'id': f'unseen-{i:02d}'} for i, r in enumerate([r for r in filter_records(corpus['dev']) if r['text'].lower() not in used][:3])]
clozes = {r['id']: cloze_of(r) for r in unseen_records}
cloze, gold = clozes[unseen_records[0]['id']]
pair_texts = [test_records[0]['text'], test_records[1]['text']]
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_BATCH': MAX_BATCH, 'MAX_TOP_K': MAX_TOP_K, 'VOCAB_SIZE': VOCAB_SIZE, 'HIDDEN_SIZE': HIDDEN_SIZE, 'MASK_TOKEN': MASK_TOKEN, 'MASK_TOKEN_ID': MASK_TOKEN_ID, 'POOLINGS': POOLINGS, 'DEFAULT_TOP_K': DEFAULT_TOP_K}
print(ceilings)
input_manifest = validate_inputs([cloze, *pair_texts], top_k=TOP_K, pooling=POOLING, names=['cloze', 'pair-a', 'pair-b'])
try:
    validate_inputs([f'A {MASK_TOKEN} and another {MASK_TOKEN}.'], top_k=TOP_K, pooling=POOLING)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'two-mask-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bert_masked_lm_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
filled = pipe.fill_mask(cloze, top_k=TOP_K)
fill_seconds = round(time.perf_counter() - started, 3)
started = time.perf_counter()
embedding_result = pipe.embed(pair_texts, pooling=POOLING)
embed_seconds = round(time.perf_counter() - started, 3)
vectors = np.asarray(embedding_result['embeddings'], dtype=np.float32)
scores = [c['score'] for c in filled['candidates']]
checks = {
    'top_k_candidates_returned': len(filled['candidates']) == TOP_K,
    'scores_descending_in_unit_interval': all(a >= b for a, b in zip(scores, scores[1:], strict=False)) and all(0.0 < s <= 1.0 for s in scores),
    'tokens_within_vocab': all(0 <= c['token_id'] < VOCAB_SIZE for c in filled['candidates']),
    'n_tokens_within_ceiling': 1 <= filled['n_tokens'] <= MAX_TEXT_TOKENS and all(1 <= n <= MAX_TEXT_TOKENS for n in embedding_result['n_tokens']),
    'one_unit_vector_per_text': vectors.shape == (2, HIDDEN_SIZE) and bool(np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-4)),
    'pooling_as_requested': embedding_result['pooling'] == POOLING and embedding_result['dim'] == HIDDEN_SIZE,
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
print({'cloze': cloze, 'gold_word': gold, 'candidates': [(c['token'], round(c['score'], 4)) for c in filled['candidates']], 'gold_in_top_k': gold in [c['token'] for c in filled['candidates']], 'seconds': fill_seconds})
print({'pair_cosine': round(float(vectors[0] @ vectors[1]), 4), 'pooling': POOLING, 'n_tokens': embedding_result['n_tokens'], 'seconds': embed_seconds, 'checks': checks, 'findings': len(input_manifest['findings']), 'score_semantics': 'softmax ranking signal, not a calibrated probability; no threshold shipped'})
before = {rid: [c['token'] for c in pipe.fill_mask(text, top_k=TOP_K)['candidates']] for rid, (text, _gold) in clozes.items()}
vectors_before = vectors
print({'unseen_clozes_filled_by_the_frozen_model': len(before)})

## 6. The unigram floor and the frozen model's masked-token metrics on the test split

Two numbers frame the adaptation. `pipe.unigram_baseline` fits an add-one-smoothed unigram model over the 30,522-token vocabulary on the training tokens and predicts every masked position of the test abstracts with it: the perplexity and top-k accuracy a model that knows the domain's word frequencies but ignores every context reaches — expect a perplexity in the thousands and a top-1 accuracy of a few percent (the commonest tokens). `pipe.evaluate` masks a seeded 15 % of each test abstract's interior WordPiece tokens (`record_seed` makes the positions a function of the record id and `SEED`, so every model scored here sees the same masks), runs the frozen encoder and masked-LM head once per record, and reads the negative log-likelihood and the rank of the original token at every masked position: **masked perplexity** (`exp` of the mean NLL), **bits per masked token** and **top-1 / top-5 accuracy**. Records over `MAX_TEXT_TOKENS` are refused, never truncated. The build record saw the frozen model near 15 with top-1 accuracy just above 50 % on these abstracts; about seven seconds on CPU.

In [ ]:
SEED = 0  # @param {type:"integer"}
MASK_RATE = 0.15  # @param {type:"number"}

def brief(m):
    return {'perplexity': round(m['perplexity'], 2), 'bits_per_token': round(m['bits_per_token'], 3), 'top1_accuracy': round(m['top1_accuracy'], 4), 'top5_accuracy': round(m['top5_accuracy'], 4), 'n_masked': m['n_masked']}

t0 = time.perf_counter()
unigram = pipe.unigram_baseline(train_records, test_records, mask_rate=MASK_RATE, seed=SEED)
print({'unigram_floor': brief(unigram), 'baseline': unigram['baseline'], 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, mask_rate=MASK_RATE, seed=SEED)
print({'frozen_model_test': brief(frozen_test), 'n_records': frozen_test['n_records'], 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
assert frozen_test['n_masked'] == unigram['n_masked'] and frozen_test['perplexity'] < unigram['perplexity']

## 7. Bounded continued masked-language-model training

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` encoder layers — four by default, 28,351,488 of 109,514,298 parameters; the word, position and token-type embeddings, the earlier layers and the masked-LM head with its decoder tied to the word embeddings stay frozen — with the model's own pre-training objective: every epoch re-draws a seeded `MASK_RATE` of each training abstract's interior tokens, replaces them with `[MASK]` and applies cross-entropy at those positions only (100 % `[MASK]`, not the upstream 80/10/10 replacement mix), AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation metrics under the evaluation masking; every epoch is scored the same way, and the epoch with the lowest validation masked perplexity is kept.

Watch validation perplexity fall from about 13 by a point or two over two epochs while top-1 accuracy creeps up (about 102 s of training plus a validation pass per epoch on CPU). The build record's sweep on this sample: two layers at 1e-4 reached 12.96, four layers at 1e-4 reached 12.51 and four layers at 5e-5 reached 12.59 in about the same time — the default keeps the standard 5e-5.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 4  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_perplexity'] = round(entry['val']['perplexity'], 2)
        row['val_top1'] = round(entry['val']['top1_accuracy'], 4)
        row['val_top5'] = round(entry['val']['top5_accuracy'], 4)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, mask_rate=MASK_RATE, seed=SEED, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'objective': adapt_result['objective'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'train_tokens': adapt_result['n_train_tokens'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no abstract in it appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6 — same masks, same seed — and the three rows are put side by side. Look for a masked perplexity a point or two below the frozen one with top-1 and top-5 accuracy up by about a point, all far from the unigram floor; the cell asserts the adapted perplexity is lower than the frozen. One hundred abstracts from one seeded split of one corpus give no dispersion estimate; the delta is sample-sanity evidence that the adaptation contract works, not a benchmark, and a lower masked perplexity on paper abstracts says nothing about your corpus until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, mask_rate=MASK_RATE, seed=SEED)
adapted_val = pipe.evaluate(val_records, mask_rate=MASK_RATE, seed=SEED)
comparison = {
    metric: {'unigram_floor': round(unigram[metric], 4), 'frozen': round(frozen_test[metric], 4), 'adapted': round(adapted_test[metric], 4)}
    for metric in ('perplexity', 'bits_per_token', 'top1_accuracy', 'top5_accuracy')
}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 4) for metric in ('perplexity', 'bits_per_token', 'top1_accuracy', 'top5_accuracy')}
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'masking': {'mask_rate': MASK_RATE, 'seed': SEED, 'n_masked_test': frozen_test['n_masked']},
    'baselines': {'unigram_floor': unigram},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/bert_masked_lm_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['perplexity'] < frozen_test['perplexity']
print({'report': 'outputs/bert_masked_lm_evaluation_report.json'})

## 9. Clozes and embeddings before and after, export the adapter and reload it

The three unseen clozes filled by the frozen model in Section 5 are filled again by the adapted model through the same `fill_mask` contract, printed side by side with the gold word and whether it appears in the top-`k` before and after; the two test abstracts are re-embedded and their cosine similarity and each vector's cosine to its frozen self are printed. Read these as observations: the adapter moves the last encoder layers, so **every embedding changes** — how much, and whether for the better on your similarity task, nothing here measures. The single-input `evaluation_report` helper — the inference-stage helper — is written for the first cloze and stays `not-measurable`, because one cloze has no metric.

`pipe.save_artifact` writes the trained tensors — the last four encoder layers, about 113 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `BERTMaskedLMPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an encoder-layer tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts an identical test perplexity on ten records, identical candidates and identical embeddings (VER4).

In [ ]:
import csv
import shutil

rows = []
for record in unseen_records:
    text, gold_word = clozes[record['id']]
    after = [c['token'] for c in pipe.fill_mask(text, top_k=TOP_K)['candidates']]
    rows.append({'id': record['id'], 'cloze': text, 'gold_word': gold_word, 'frozen_candidates': ' '.join(before[record['id']]), 'adapted_candidates': ' '.join(after), 'gold_in_top_k_frozen': gold_word in before[record['id']], 'gold_in_top_k_adapted': gold_word in after})
    print({k: rows[-1][k] for k in ('id', 'gold_word', 'frozen_candidates', 'adapted_candidates', 'gold_in_top_k_frozen', 'gold_in_top_k_adapted')})
vectors_after = np.asarray(pipe.embed(pair_texts, pooling=POOLING)['embeddings'], dtype=np.float32)
embedding_shift = {'pair_cosine_frozen': round(float(vectors_before[0] @ vectors_before[1]), 4), 'pair_cosine_adapted': round(float(vectors_after[0] @ vectors_after[1]), 4), 'self_cosine_frozen_vs_adapted': [round(float(vectors_before[i] @ vectors_after[i]), 4) for i in range(2)]}
single_report = evaluation_report(pipe.fill_mask(cloze, top_k=TOP_K), [gold], sample_kind='one unseen SciTLDR abstract cloze' if not USE_BYOD else 'one BYOD test record')
print({'embedding_shift': embedding_shift, 'single_input_report_verdict': single_report['verdict'], 'clozes_with_changed_candidates': sum(r['frozen_candidates'] != r['adapted_candidates'] for r in rows), 'of': len(rows)})
with open('outputs/bert_masked_lm_cloze.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

artifact_dir = Path('outputs/bert_masked_lm_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'bert_masked_lm', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = BERTMaskedLMPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
parity_records = test_records[:10]
ppl_pair = (pipe.evaluate(parity_records, mask_rate=MASK_RATE, seed=SEED)['perplexity'], reloaded.evaluate(parity_records, mask_rate=MASK_RATE, seed=SEED)['perplexity'])
cand_pair = [(r['adapted_candidates'], ' '.join(c['token'] for c in reloaded.fill_mask(r['cloze'], top_k=TOP_K)['candidates'])) for r in rows]
vectors_reloaded = np.asarray(reloaded.embed(pair_texts, pooling=POOLING)['embeddings'], dtype=np.float32)
parity = {'perplexity_in_memory': round(ppl_pair[0], 6), 'perplexity_reloaded': round(ppl_pair[1], 6), 'identical_candidates': sum(a == b for a, b in cand_pair), 'of': len(cand_pair), 'embeddings_identical': bool(np.array_equal(vectors_after, vectors_reloaded))}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert abs(ppl_pair[0] - ppl_pair[1]) < 1e-6 and parity['identical_candidates'] == parity['of'] and parity['embeddings_identical']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'cloze': cloze, 'gold_word': gold, 'fill_mask': {k: filled[k] for k in ('candidates', 'top_k', 'n_tokens')}, 'embed': {k: embedding_result[k] for k in ('dim', 'pooling', 'normalized', 'n_tokens')}, 'seconds': {'fill_mask': fill_seconds, 'embed': embed_seconds}},
    'comparison': comparison,
    'clozes_before_after': rows,
    'embedding_shift': embedding_shift,
    'single_input_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/bert_masked_lm_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen encoder already predicts the masked words of paper abstracts far better than a unigram model does (a masked perplexity near 15 against a floor in the thousands, top-1 accuracy above 50 % against a few percent — BooksCorpus and Wikipedia contain technical prose), and a bounded continuation of its own pre-training objective on 300 abstracts, touching only the last four encoder layers, lowers held-out masked perplexity by a few points in a few minutes on CPU, with a 113 MB adapter that reloads to identical likelihoods, candidates and embeddings. That is the claim: the adaptation contract can continue pre-training end to end on a real corpus, and the number it produces is read against the frozen model and a context-free floor rather than in isolation.

Masked perplexity is intrinsic: it says how well the encoder predicts missing tokens of text it did not write, at one seeded 15 % masking of one seeded split of one corpus with no dispersion estimate. It is not embedding quality, retrieval performance or downstream accuracy, and a lower value does not make the candidates or the cosine table in Section 9 better — those are observations. The adapter changes the last layers, which every input shares, so every embedding shifts (Section 9 prints each vector's cosine to its frozen self) and the fill-mask `score` stays a softmax ranking signal, not a probability. Text is lower-cased and accent-stripped by the tokenizer; texts over 512 WordPiece tokens are rejected everywhere, never truncated; the checkpoint carries the gender and occupation associations the upstream card documents, and continued pre-training on a corpus does not remove them.

Three things to carry to real data. **Floors first:** the unigram floor and the frozen masked perplexity on *your* held-out documents are the numbers to read before any adapted one. **Leakage:** de-duplicate texts across splits (the contract does this case-insensitively) and split by document collection or author when your documents come from one. **Downstream:** a better masked perplexity on your domain is a reason to *measure* your similarity or classification task with the adapted encoder, not evidence that it improved.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real corpus, validate the demonstrated dataset contract without leakage, execute both inference contracts and a bounded continued pre-training, evaluate by masked-token metrics against a trivial floor and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, embedding or cloze quality on any domain, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 1` and watch the gain shrink; set `EPOCHS = 4` and watch whether validation perplexity keeps falling or turns (the best epoch is kept either way); change `SEED` in Section 6 and read how much the masked metrics move with a different set of masked positions; switch `POOLING` and re-read the cosine shift; or bring your own documents through BYOD and read the unigram floor before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bert-masked-lm-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google-bert/bert-base-uncased
- Upstream code: https://github.com/google-research/bert
- BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding (Devlin et al., NAACL 2019): https://arxiv.org/abs/1810.04805
- TLDR: Extreme Summarization of Scientific Documents (Cachola et al., EMNLP Findings 2020; SciTLDR, Apache-2.0): https://arxiv.org/abs/2004.15011
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)